# Part 4: LLM-Powered Feature (Model Prediction Explanation Pipeline)

This notebook implements a customer churn prediction explanation pipeline. It loads our best-performing model, makes predictions and probabilities for three customer profiles, passes those values to the Google Gemini API (or any OpenAI-compatible API), and validates the returned explanations against a strict JSON schema while checking for PII leaks.

We have implemented the following tasks step-by-step to build a robust pipeline.

### Step 1: Libraries, configuration, and JSON schema definition

We load necessary Python packages like `requests`, `joblib`, `pandas`, and `jsonschema`. We also define a JSON schema with 5 required scalar fields to validate the output format of our LLM response.

In [ ]:
import os
import re
import json
import requests
import joblib
import time
import pandas as pd
import numpy as np
from jsonschema import validate, ValidationError

# Load environment variables from a local .env file (if it exists)
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("Local environment variables loaded from .env file.")
except ImportError:
    print("dotenv library is not installed. Skipping .env loading.")

# Define the strict schema to validate the LLM's explanation output.
# The schema includes exactly 5 required scalar fields.
EXPLANATION_SCHEMA = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": ["prediction_label", "confidence_level", "top_reason", "second_reason", "next_step"]
}

print("Step 1 completed: Libraries and schema loaded.")

### Step 2: Reusable LLM Call Function

This function, `call_llm`, sends system and user prompts to the LLM API using the `requests` library. It reads the API key from environment variables or Google Colab secrets. To be flexible, it automatically adapts to Google Gemini's native API structure or standard OpenAI-compatible API structures (like OpenRouter) based on the target URL.

**Note on Truncation**: We set `max_tokens=2048` by default. Under models like Gemini 2.5, reasoning/thinking tokens count against the output limit. Increasing this limit prevents JSON responses from being truncated mid-flight.

In [ ]:
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=2048):
    """
    Sends system and user prompts to the LLM API and returns the generated text.
    """
    # Get the API Key
    api_key = None
    try:
        from google.colab import userdata
        api_key = userdata.get('LLM_API_KEY')
    except (ImportError, Exception):
        api_key = os.environ.get("LLM_API_KEY")
        
    if not api_key:
        print("[ERROR] LLM_API_KEY is not set. Please set it in your environment or Colab secrets.")
        return None
        
    # Get URL and model from environment, defaulting to standard Gemini endpoint
    api_url = os.environ.get(
        "LLM_API_URL", 
        "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent"
    )
    model_name = os.environ.get("LLM_MODEL", "gemini-2.5-flash")
    
    # Format and make request depending on the endpoint type
    if "googleapis.com" in api_url:
        # Google Gemini native REST payload structure
        if "key=" not in api_url:
            separator = "&" if "?" in api_url else "?"
            request_url = f"{api_url}{separator}key={api_key}"
        else:
            request_url = api_url
            
        headers = {"Content-Type": "application/json"}
        payload = {
            "systemInstruction": {
                "parts": [{"text": system_prompt}]
            },
            "contents": [
                {"parts": [{"text": user_prompt}]}
            ],
            "generationConfig": {
                "temperature": temperature,
                "maxOutputTokens": max_tokens
            }
        }
        
        try:
            response = requests.post(request_url, headers=headers, json=payload)
            if response.status_code == 200:
                response_json = response.json()
                return response_json['candidates'][0]['content']['parts'][0]['text']
            else:
                print(f"Gemini API call failed with status: {response.status_code}")
                print("API Error Output:", response.text)
                return None
        except Exception as e:
            print("Exception occurred during Gemini call:", str(e))
            return None
            
    else:
        # Standard OpenAI / OpenRouter payload structure
        headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": model_name,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "temperature": temperature,
            "max_tokens": max_tokens
        }
        
        try:
            response = requests.post(api_url, headers=headers, json=payload)
            if response.status_code == 200:
                response_json = response.json()
                return response_json['choices'][0]['message']['content']
            else:
                print(f"OpenAI/OpenRouter call failed with status: {response.status_code}")
                print("API Error Output:", response.text)
                return None
        except Exception as e:
            print("Exception occurred during OpenAI/OpenRouter call:", str(e))
            return None

### Step 3: Simple connection verification test

Let's test our `call_llm` function with a basic request requesting a simple reply 'hello'.

In [ ]:
test_sys = "You are a helpful assistant. Reply with only the word: hello"
test_usr = "hello"

print("Testing API Connection...")
res = call_llm(test_sys, test_usr, temperature=0.0)
print("Raw Response:", res)

if res and "hello" in res.lower():
    print("API Key Connection Successful!")
else:
    print("Could not verify connection. Check your API key.")

### Step 4: PII guardrail regex implementation and testing

We implement a safety checker using regular expressions to screen prompts for emails or telephone numbers before querying the LLM. If PII is detected, the process halts.

In [ ]:
def has_pii(text):
    """
    Returns True if the text contains a standard email or phone number.
    """
    email_regex = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_regex = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    
    if re.search(email_regex, text) or re.search(phone_regex, text):
        return True
    return False

# Test 1: Contains PII (Email)
pii_email_input = "Customer email is sivarajan@test.com. Churn predicted: Yes."
# Test 2: Contains No PII
clean_input = "Customer on Contract: Month-to-month, tenure: 2 months."

print("--- PII Guardrail Test 1 (PII Input) ---")
print("Input:", pii_email_input)
if has_pii(pii_email_input):
    print("Result: Input blocked: PII detected.")
else:
    print("Result: Proceeding to LLM call...")

print("\n--- PII Guardrail Test 2 (Clean Input) ---")
print("Input:", clean_input)
if has_pii(clean_input):
    print("Result: Input blocked: PII detected.")
else:
    print("Result: Input passed. Calling LLM...")
    test_resp = call_llm("You are a helpful assistant.", clean_input, temperature=0.0)
    print("LLM Response:", test_resp)

### Step 5: Feature preprocessing helper for the machine learning model

We construct a helper function `encode_record` to format a raw input dictionary of features so that it perfectly matches the exact columns and one-hot encoding representation that our ML model expects.

In [ ]:
def encode_record(features_dict, training_columns):
    """
    Encodes a single raw record dict to a DataFrame aligned with the model features.
    """
    # Convert record dict to DataFrame
    df = pd.DataFrame([features_dict])
    
    # Map ordinal columns
    contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
    if 'Contract' in df.columns:
        df['Contract'] = df['Contract'].map(contract_map)
        
    internet_map = {'No': 0, 'DSL': 1, 'Fiber optic': 2}
    if 'InternetService' in df.columns:
        df['InternetService'] = df['InternetService'].map(internet_map)
        
    # Nominal columns to encode
    nominal_cols = [
        'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 
        'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'PaymentMethod'
    ]
    
    # One-hot encode existing nominal columns
    existing_nominal = [c for c in nominal_cols if c in df.columns]
    df_encoded = pd.get_dummies(df, columns=existing_nominal, drop_first=True)
    
    # Align columns to training columns
    for col in training_columns:
        if col not in df_encoded.columns:
            df_encoded[col] = 0
            
    # Convert boolean variables to integer
    for col in df_encoded.columns:
        if df_encoded[col].dtype == bool:
            df_encoded[col] = df_encoded[col].astype(int)
            
    # Reorder and restrict columns to match model training feature set
    df_encoded = df_encoded[training_columns]
    return df_encoded

### Step 6: Load model and setup explanation prompts

We load our saved model using `joblib.load('best_model.pkl')` and define our handcrafted customer profiles. We also draft the verbatim system prompt instructing the LLM to output a clean JSON prediction explanation matching our required schema.

In [ ]:
# Load best_model.pkl
model_file = "best_model.pkl"
print("Loading classification model:", model_file)
best_model = joblib.load(model_file)

# Extract features list expected by model
training_features = list(best_model.feature_names_in_)
print(f"Loaded successfully. Model expects {len(training_features)} features.")

# Create three handcrafted customer profiles representing different types of contracts and churn risks
three_profiles = [
    {
        "gender": "Female", "SeniorCitizen": 1, "Partner": "No", "Dependents": "No",
        "tenure": 2, "PhoneService": "Yes", "MultipleLines": "No", "InternetService": "Fiber optic",
        "OnlineSecurity": "No", "OnlineBackup": "No", "DeviceProtection": "No", "TechSupport": "No",
        "StreamingTV": "Yes", "StreamingMovies": "Yes", "Contract": "Month-to-month",
        "PaperlessBilling": "Yes", "PaymentMethod": "Electronic check", "TotalCharges": 170.0
    },
    {
        "gender": "Male", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "Yes",
        "tenure": 60, "PhoneService": "Yes", "MultipleLines": "No", "InternetService": "DSL",
        "OnlineSecurity": "Yes", "OnlineBackup": "Yes", "DeviceProtection": "Yes", "TechSupport": "Yes",
        "StreamingTV": "No", "StreamingMovies": "No", "Contract": "Two year",
        "PaperlessBilling": "No", "PaymentMethod": "Mailed check", "TotalCharges": 1200.0
    },
    {
        "gender": "Female", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "No",
        "tenure": 12, "PhoneService": "Yes", "MultipleLines": "Yes", "InternetService": "DSL",
        "OnlineSecurity": "No", "OnlineBackup": "Yes", "DeviceProtection": "No", "TechSupport": "No",
        "StreamingTV": "No", "StreamingMovies": "Yes", "Contract": "Month-to-month",
        "PaperlessBilling": "Yes", "PaymentMethod": "Electronic check", "TotalCharges": 480.0
    }
]

# Verbatim System Prompt defining model role and expected schema
sys_prompt_explanation = (
    "You are an expert customer churn analyst. Explain the machine learning model's churn prediction "
    "for a specific customer based on their feature values, predicted class, and probability. "
    "You must output ONLY a valid JSON object matching this schema:\n"
    "{\n"
    "  \"prediction_label\": \"A short string summarizing the risk status\",\n"
    "  \"confidence_level\": \"low|medium|high\",\n"
    "  \"top_reason\": \"Primary driver behind this model prediction\",\n"
    "  \"second_reason\": \"Secondary contributing factor to the prediction\",\n"
    "  \"next_step\": \"Recommended action for customer retention or service maintenance\"\n"
    "}\n"
    "Do not include any markdown formatting, backticks, or text outside the JSON object. Output ONLY valid JSON.\n"
    "Keep all descriptions/reasons extremely concise (single sentences) to ensure it fits response limits."
)

print("Setup Complete.")

### Step 7: Run temperature A/B comparisons

We evaluate Customer Profile 1 at `temperature=0.0` (for deterministic and highly structured responses) and `temperature=0.7` (to observe differences in phrasing and token selection).

In [ ]:
# Prep Customer Profile 1 details
cust1 = three_profiles[0]
cust1_encoded = encode_record(cust1, training_features)

# Run Model Inference
pred_class1 = int(best_model.predict(cust1_encoded)[0])
pred_prob1 = float(best_model.predict_proba(cust1_encoded)[0][1])

# Construct User Prompt
user_prompt_compare = (
    "Customer details:\n"
    f"- Contract type: {cust1['Contract']}\n"
    f"- Internet service: {cust1['InternetService']}\n"
    f"- Tenure duration: {cust1['tenure']} months\n"
    f"- Total Charges: ${cust1['TotalCharges']}\n"
    f"- Payment Method: {cust1['PaymentMethod']}\n\n"
    f"Model Churn Prediction: Class {pred_class1}\n"
    f"Model Churn Probability: {round(pred_prob1, 4)}"
)

print("--- Run at temperature = 0.0 ---")
out_t0 = call_llm(sys_prompt_explanation, user_prompt_compare, temperature=0.0)
print(out_t0)

print("\nWaiting 15 seconds to avoid hitting API rate limits...")
time.sleep(15)

print("\n--- Run at temperature = 0.7 ---")
out_t7 = call_llm(sys_prompt_explanation, user_prompt_compare, temperature=0.7)
print(out_t7)

### Step 8: End-to-end batch explanation pipeline for all three profiles

We run all three customer profiles through our model, perform inference, apply PII guardrails, send clean prompts to the LLM, clean markdown symbols, validate response correctness against our JSON schema using `jsonschema.validate()`, and handle any errors by writing a fallback dictionary containing null parameters.

In [ ]:
results_records = []

for index, cust in enumerate(three_profiles):
    print(f"\nProcessing Customer Profile {index + 1}...")
    
    # Preprocess feature dict
    encoded_df = encode_record(cust, training_features)
    p_class = int(best_model.predict(encoded_df)[0])
    p_prob = float(best_model.predict_proba(encoded_df)[0][1])
    
    # Create prompt template
    user_prompt_profile = (
        "Customer details:\n"
        f"- Contract type: {cust['Contract']}\n"
        f"- Internet service: {cust['InternetService']}\n"
        f"- Tenure duration: {cust['tenure']} months\n"
        f"- Total Charges: ${cust['TotalCharges']}\n"
        f"- Payment Method: {cust['PaymentMethod']}\n\n"
        f"Model Churn Prediction: Class {p_class}\n"
        f"Model Churn Probability: {round(p_prob, 4)}"
    )
    
    # Check PII Guardrail
    if has_pii(user_prompt_profile):
        print("Input blocked: PII detected.")
        results_records.append({
            "input": user_prompt_profile,
            "pred_class": p_class,
            "probability": p_prob,
            "output": None,
            "status": "Blocked"
        })
        continue
        
    # Make LLM Call at temperature 0.0
    print("Requesting explanation from API...")
    raw_out = call_llm(sys_prompt_explanation, user_prompt_profile, temperature=0.0)
    
    # Strip surrounding markdown backticks if returned
    cleaned_out = ""
    if raw_out:
        cleaned_out = raw_out.strip()
        if cleaned_out.startswith("```json"):
            cleaned_out = cleaned_out[7:]
        if cleaned_out.endswith("```"):
            cleaned_out = cleaned_out[:-3]
        cleaned_out = cleaned_out.strip()
        
    # Parse and Validate against schema
    is_valid = False
    validated_obj = None
    err_message = ""
    
    if cleaned_out:
        try:
            validated_obj = json.loads(cleaned_out)
            validate(instance=validated_obj, schema=EXPLANATION_SCHEMA)
            is_valid = True
            print("Validation Outcome: PASS")
        except json.JSONDecodeError as de:
            err_message = f"JSON Decode Error: {de}"
            print("Validation Outcome: FAIL -", err_message)
        except ValidationError as ve:
            err_message = f"Schema ValidationError: {ve.message}"
            print("Validation Outcome: FAIL -", err_message)
    else:
        err_message = "LLM returned empty or None response."
        print("Validation Outcome: FAIL -", err_message)
        
    # Fallback Mechanism
    if not is_valid:
        validated_obj = {
            "prediction_label": None,
            "confidence_level": None,
            "top_reason": None,
            "second_reason": None,
            "next_step": f"Fallback applied. Error message: {err_message}"
        }
        print("Fallback object applied.")
        
    # Print outcomes
    print("Final validated assessment:")
    print(json.dumps(validated_obj, indent=2))
    
    results_records.append({
        "input": cust,
        "pred_class": p_class,
        "probability": p_prob,
        "output": validated_obj,
        "status": "Passed - Pass" if is_valid else "Passed - Fail"
    })
    
    print("Waiting 15 seconds to respect rate limits...")
    time.sleep(15)